# 🌡️ Thermal Comfort Visualisation System

Esplora il **comfort termico percepito** (UTCI) per qualsiasi luogo del mondo,
confronta più città, analizza il clima storico e visualizza previsioni vs realtà.

---

## Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

try:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', DeprecationWarning)
        get_ipython().run_line_magic('matplotlib', 'widget')
except Exception:
    get_ipython().run_line_magic('matplotlib', 'inline')

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd

from thermalcomfort import ThermalComfortSystem, ComfortParams
from thermalcomfort.climate import ClimateAnalysis
from thermalcomfort.comfort.indices import UTCI_CATEGORIES, UTCI_COLORS
from thermalcomfort.locations import LOCATIONS
from thermalcomfort.mapview import build_map_from_locations, plot_map_static

def add_utci_background(ax):
    for cat, (lo, hi) in UTCI_CATEGORIES.items():
        ax.axhspan(lo, hi, alpha=0.09, color=UTCI_COLORS[cat], lw=0)

# Preset di esposizione solare — sun_exposure è una frazione continua [0, 1]
# (0 = ombra piena, 1 = pieno sole) usata dal modello di MRT radiativo;
# qui la esponiamo come scelte qualitative invece di uno slider numerico
# poco intuitivo.
SUN_EXPOSURE_OPTIONS = [
    ('☀️ Pieno sole', 1.0),
    ('🌤️ Esposizione parziale (es. passeggiata in città)', 0.5),
    ('🌳 Ombra leggera (albero isolato, tenda)', 0.2),
    ('🌲 Ombra fitta (pineta, portico, vicolo stretto)', 0.05),
]

# Superficie del terreno: influenza la componente riflessa a onda corta e
# l'emissione a onda lunga del suolo nel bilancio radiativo della MRT.
SURFACE_TYPE_OPTIONS = [
    ('🛣️ Asfalto / cemento (urbano)', 'asphalt'),
    ('🌱 Prato / vegetazione', 'grass'),
]

def make_scenario_widgets(width='320px'):
    """Coppia di widget (esposizione solare, superficie) condivisa dalle sezioni."""
    w_sun = widgets.Dropdown(
        options=SUN_EXPOSURE_OPTIONS, value=0.5,
        description='Ombra:', style={'description_width': '80px'},
        layout=widgets.Layout(width=width),
    )
    w_surf = widgets.Dropdown(
        options=SURFACE_TYPE_OPTIONS, value='asphalt',
        description='Superficie:', style={'description_width': '80px'},
        layout=widgets.Layout(width=width),
    )
    return w_sun, w_surf

tcs = ThermalComfortSystem()
print('✓ Sistema pronto')

✓ Sistema pronto


---
## 📍 Luoghi predefiniti

Puoi usarli nelle celle successive o definire i tuoi.

In [2]:
print(f'Definiti {len(LOCATIONS)} luoghi (da thermalcomfort.locations)')

Definiti 48 luoghi (da thermalcomfort.locations)


---
## 🔥 Sezione 1 — Serie temporale interattiva

Seleziona luogo, periodo, esposizione solare e superficie.

In [3]:
w_loc1 = widgets.Dropdown(
    options=list(LOCATIONS.keys()), value='Firenze',
    description='Luogo:', style={'description_width': '80px'}, layout=widgets.Layout(width='260px')
)
w_start1 = widgets.DatePicker(
    value=pd.Timestamp.now().date() - pd.Timedelta(days=7),
    description='Inizio:', style={'description_width': '80px'}, layout=widgets.Layout(width='220px')
)
w_end1 = widgets.DatePicker(
    value=pd.Timestamp.now().date(),
    description='Fine:', style={'description_width': '80px'}, layout=widgets.Layout(width='220px')
)
w_sun1, w_surf1 = make_scenario_widgets()
w_tz1 = widgets.Text(value='Europe/Rome', description='Timezone:',
    style={'description_width': '80px'}, layout=widgets.Layout(width='220px')
)
btn1 = widgets.Button(description='▶ Aggiorna', button_style='primary',
    layout=widgets.Layout(width='130px')
)
out1 = widgets.Output()

def run1(_):
    with out1:
        clear_output(wait=True)
        loc = LOCATIONS[w_loc1.value]
        params = ComfortParams(sun_exposure=w_sun1.value, surface_type=w_surf1.value)
        try:
            df = tcs.get(loc, str(w_start1.value), str(w_end1.value) + ' 23:00', params=params)
            fig = tcs.plot(df, title=str(loc), local_tz=w_tz1.value or None, show=False)
            plt.show()
            utci = df['utci'].dropna()
            cats = df['utci_category']
            print(f"UTCI: media {utci.mean():.1f}°C, min {utci.min():.1f}°C, max {utci.max():.1f}°C")
            print(f"Ore senza stress: {(cats=='no thermal stress').mean():.0%}  "
                  f"| caldo: {cats.str.contains('heat stress').mean():.0%}  "
                  f"| freddo: {cats.str.contains('cold stress').mean():.0%}")
        except Exception as e:
            print(f'Errore: {e}')

btn1.on_click(run1)
display(widgets.VBox([
    widgets.HBox([w_loc1, w_start1, w_end1]),
    widgets.HBox([w_sun1, w_surf1, w_tz1]),
    btn1,
    out1,
]))

---
## 📊 Sezione 2 — Confronto tra luoghi

Confronta UTCI (o altra variabile) tra più città nello stesso periodo.

In [4]:
w_locs2 = widgets.SelectMultiple(
    options=list(LOCATIONS.keys()),
    value=['Firenze', 'Livorno', 'Borgo San Lorenzo'],
    description='Luoghi:', rows=8,
    style={'description_width': '80px'}, layout=widgets.Layout(width='280px')
)
w_start2 = widgets.DatePicker(
    value=pd.Timestamp.now().date() - pd.Timedelta(days=7),
    description='Inizio:', style={'description_width': '80px'}
)
w_end2 = widgets.DatePicker(
    value=pd.Timestamp.now().date(),
    description='Fine:', style={'description_width': '80px'}
)
w_var2 = widgets.Dropdown(
    options=['utci','temperature_2m','heat_index','wind_chill','wbgt_outdoor','mrt'],
    value='utci', description='Variabile:', style={'description_width': '80px'}
)
w_sun2, w_surf2 = make_scenario_widgets()
w_tz2 = widgets.Text(value='Europe/Rome', description='Timezone:',
    style={'description_width': '80px'}
)
btn2 = widgets.Button(description='▶ Confronta', button_style='primary')
out2 = widgets.Output()

def run2(_):
    with out2:
        clear_output(wait=True)
        selected = [LOCATIONS[n] for n in w_locs2.value]
        if len(selected) < 2:
            print('Seleziona almeno 2 luoghi (Ctrl+click)')
            return
        params = ComfortParams(sun_exposure=w_sun2.value, surface_type=w_surf2.value)
        try:
            fig = tcs.compare(selected, str(w_start2.value), str(w_end2.value) + ' 23:00',
                              variable=w_var2.value, params=params,
                              local_tz=w_tz2.value or None, show=False)
            if w_var2.value == 'utci' and fig.axes:
                add_utci_background(fig.axes[0])
            plt.show()
        except Exception as e:
            print(f'Errore: {e}')

btn2.on_click(run2)
display(widgets.VBox([
    widgets.HBox([
        w_locs2,
        widgets.VBox([w_start2, w_end2, w_var2, w_sun2, w_surf2, w_tz2]),
    ]),
    btn2, out2,
]))

---
## 🗓️ Sezione 3 — Profilo climatologico

Media storica (2010–2023) mese per mese, o profilo orario per un mese specifico.

In [5]:
MONTHS = ['Tutti (grafico annuale)', 'Gen','Feb','Mar','Apr','Mag','Giu',
          'Lug','Ago','Set','Ott','Nov','Dic']

w_loc3 = widgets.Dropdown(options=list(LOCATIONS.keys()), value='Firenze',
    description='Luogo:', style={'description_width': '80px'},
    layout=widgets.Layout(width='260px')
)
w_month3 = widgets.Dropdown(options=MONTHS, value='Tutti (grafico annuale)',
    description='Mese:', style={'description_width': '80px'},
    layout=widgets.Layout(width='260px')
)
w_sy3 = widgets.IntSlider(value=2015, min=2000, max=2026, step=1,
    description='Dal:', style={'description_width': '60px'},
    layout=widgets.Layout(width='320px')
)
w_ey3 = widgets.IntSlider(value=2025, min=2001, max=2026, step=1,
    description='Al:', style={'description_width': '60px'},
    layout=widgets.Layout(width='320px')
)
w_sun3, w_surf3 = make_scenario_widgets()
btn3 = widgets.Button(description='▶ Calcola', button_style='warning')
out3 = widgets.Output()

def run3(_):
    with out3:
        clear_output(wait=True)
        loc = LOCATIONS[w_loc3.value]
        params = ComfortParams(sun_exposure=w_sun3.value, surface_type=w_surf3.value)
        ca = ClimateAnalysis(start_year=w_sy3.value, end_year=w_ey3.value)
        print(f'Caricamento dati per {loc} ({w_sy3.value}–{w_ey3.value}) — può richiedere qualche minuto alla prima esecuzione …')
        try:
            month_idx = MONTHS.index(w_month3.value)
            if month_idx == 0:
                fig = ca.plot_monthly(loc, params=params, show=False)
            else:
                fig = ca.plot_hourly_profile(loc, month=month_idx, params=params, show=False)
            plt.show()
        except Exception as e:
            print(f'Errore: {e}')

btn3.on_click(run3)
display(widgets.VBox([
    widgets.HBox([w_loc3, w_month3]),
    widgets.HBox([w_sy3, w_ey3]),
    widgets.HBox([w_sun3, w_surf3]),
    btn3, out3,
]))

---
## 🏆 Sezione 4 — Classifica dei luoghi

Quale città offre il comfort migliore in un determinato mese?

In [6]:
MONTHS_RANK = ['Anno intero','Gen','Feb','Mar','Apr','Mag','Giu',
               'Lug','Ago','Set','Ott','Nov','Dic']

w_locs4 = widgets.SelectMultiple(
    options=list(LOCATIONS.keys()),
    value=['Firenze','Roma','Milano','Napoli','Parigi','Barcellona','Atene','Berlino','Londra','Stoccolma'],
    description='Luoghi:', rows=12,
    style={'description_width': '80px'}, layout=widgets.Layout(width='300px')
)
w_month4 = widgets.Dropdown(options=MONTHS_RANK, value='Lug',
    description='Mese:', style={'description_width': '80px'}
)
w_sy4 = widgets.IntSlider(value=2015, min=2000, max=2026,
    description='Dal:', style={'description_width': '60px'},
    layout=widgets.Layout(width='300px')
)
w_ey4 = widgets.IntSlider(value=2025, min=2001, max=2026,
    description='Al:', style={'description_width': '60px'},
    layout=widgets.Layout(width='300px')
)
w_sun4, w_surf4 = make_scenario_widgets(width='300px')
btn4 = widgets.Button(description='▶ Classifica', button_style='danger')
out4 = widgets.Output()

def run4(_):
    with out4:
        clear_output(wait=True)
        selected = [LOCATIONS[n] for n in w_locs4.value]
        if not selected:
            print('Seleziona almeno un luogo')
            return
        month_idx = MONTHS_RANK.index(w_month4.value)
        month = month_idx if month_idx > 0 else None
        params = ComfortParams(sun_exposure=w_sun4.value, surface_type=w_surf4.value)
        ca = ClimateAnalysis(start_year=w_sy4.value, end_year=w_ey4.value)
        print('Elaborazione …')
        try:
            df = ca.rank_locations(selected, month=month, params=params)
            print(df.to_string())
            fig = ca.plot_rank(selected, month=month, params=params, show=False)
            plt.show()
        except Exception as e:
            print(f'Errore: {e}')

btn4.on_click(run4)
display(widgets.VBox([
    widgets.HBox([w_locs4, widgets.VBox([w_month4, w_sy4, w_ey4, w_sun4, w_surf4])]),
    btn4, out4,
]))

---
## 🗺️ Sezione 5 — Mappa interattiva

Visualizza il comfort percepito su una mappa geografica a un dato momento.

In [7]:
w_locs5 = widgets.SelectMultiple(
    options=list(LOCATIONS.keys()),
    value=['Firenze','Roma','Milano','Napoli','Parigi','Barcellona','Atene','Berlino'],
    description='Luoghi:', rows=10,
    style={'description_width': '80px'}, layout=widgets.Layout(width='300px')
)
w_dt5 = widgets.Text(
    value=(pd.Timestamp.now('UTC') - pd.Timedelta(hours=2)).strftime('%Y-%m-%d %H:00'),
    description='Data/ora UTC:', style={'description_width': '110px'},
    layout=widgets.Layout(width='280px')
)
w_var5 = widgets.Dropdown(
    options=['utci','temperature_2m','heat_index'],
    value='utci', description='Variabile:', style={'description_width': '80px'}
)
w_sun5, w_surf5 = make_scenario_widgets(width='300px')
btn5_static = widgets.Button(description='Mappa statica', button_style='info')
btn5_html = widgets.Button(description='Mappa HTML (browser)', button_style='primary')
out5 = widgets.Output()

def _get_map_data(locs, dt_str, sun, surface):
    params = ComfortParams(sun_exposure=sun, surface_type=surface)
    dt = pd.Timestamp(dt_str, tz='UTC')
    start = (dt - pd.Timedelta(hours=12)).strftime('%Y-%m-%d')
    end   = (dt + pd.Timedelta(hours=12)).strftime('%Y-%m-%d')
    pairs = []
    for loc in locs:
        df = tcs.get(loc, start, end, params=params)
        pairs.append((loc, df))
    return pairs, dt

def run5_static(_):
    with out5:
        clear_output(wait=True)
        selected = [LOCATIONS[n] for n in w_locs5.value]
        try:
            pairs, dt = _get_map_data(selected, w_dt5.value, w_sun5.value, w_surf5.value)
            fig = plot_map_static(pairs, dt, variable=w_var5.value, show=False)
            plt.show()
        except Exception as e:
            print(f'Errore: {e}')

def run5_html(_):
    with out5:
        clear_output(wait=True)
        selected = [LOCATIONS[n] for n in w_locs5.value]
        try:
            pairs, dt = _get_map_data(selected, w_dt5.value, w_sun5.value, w_surf5.value)
            path = build_map_from_locations(pairs, dt, variable=w_var5.value, open_browser=True)
            print(f'Mappa salvata e aperta: {path}')
        except Exception as e:
            print(f'Errore: {e}')

btn5_static.on_click(run5_static)
btn5_html.on_click(run5_html)
display(widgets.VBox([
    widgets.HBox([w_locs5, widgets.VBox([w_dt5, w_var5, w_sun5, w_surf5])]),
    widgets.HBox([btn5_static, btn5_html]),
    out5,
]))

---
## 📡 Sezione 6 — Previsione vs Realtà

Confronta il modello di previsione ECMWF con la rianalisi ERA5 per valutare l'errore tipico.

> ⚠️ Disponibile solo per gli ultimi ~90 giorni.

In [8]:
w_loc6 = widgets.Dropdown(options=list(LOCATIONS.keys()), value='Firenze',
    description='Luogo:', style={'description_width': '80px'},
    layout=widgets.Layout(width='260px')
)
w_start6 = widgets.DatePicker(
    value=pd.Timestamp.now().date() - pd.Timedelta(days=14),
    description='Inizio:', style={'description_width': '80px'}
)
w_end6 = widgets.DatePicker(
    value=pd.Timestamp.now().date() - pd.Timedelta(days=2),
    description='Fine:', style={'description_width': '80px'}
)
w_var6 = widgets.Dropdown(
    options=['utci','temperature_2m','relative_humidity_2m','wind_speed_10m'],
    value='utci', description='Variabile:', style={'description_width': '80px'}
)
w_sun6, w_surf6 = make_scenario_widgets()
w_tz6 = widgets.Text(value='Europe/Rome', description='Timezone:',
    style={'description_width': '80px'}
)
btn6 = widgets.Button(description='▶ Analizza', button_style='primary')
out6 = widgets.Output()

def run6(_):
    with out6:
        clear_output(wait=True)
        loc = LOCATIONS[w_loc6.value]
        params = ComfortParams(sun_exposure=w_sun6.value, surface_type=w_surf6.value)
        try:
            fig = tcs.plot_forecast_vs_actual(
                loc, str(w_start6.value), str(w_end6.value) + ' 23:00',
                variable=w_var6.value, params=params,
                local_tz=w_tz6.value or None, show=False
            )
            if w_var6.value == 'utci' and fig.axes:
                add_utci_background(fig.axes[0])
            plt.show()
        except ValueError as e:
            print(f'Errore: {e}\nRidurre il periodo a meno di 90 giorni fa.')
        except Exception as e:
            print(f'Errore: {e}')

btn6.on_click(run6)
display(widgets.VBox([
    widgets.HBox([w_loc6, w_start6, w_end6]),
    widgets.HBox([w_var6, w_sun6, w_surf6, w_tz6]),
    btn6, out6,
]))